# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jh-emon002/flyrank-intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
%pip -q install duckdb huggingface_hub scikit-learn

In [5]:
import os
import duckdb
import pandas as pd

from huggingface_hub import HfApi

# Get HF token safely from Colab Secrets
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN not found. Add it in Colab Secrets."

# Verify the token itself without printing it
who = HfApi(token=HF_TOKEN).whoami()
print("Authenticated as:", who["name"])

# DuckDB connection
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("Setup complete.")

Authenticated as: jh-emon002
Setup complete.


In [1]:
FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-15"

OUTCOME_START = "2026-03-17"
OUTCOME_END = "2026-03-31"

MIN_IMPRESSIONS = 100
DECLINE_THRESHOLD = 0.80

## 1. My rule and its reason codes

### Signal checks before defining the rule

Before fixing the baseline rule, I test two signals that could reasonably
support it. I use only information available before the March 16 decision
point. The outcome used to audit the signals comes from March 17–31.

In [6]:
page_frame = con.sql(f"""
WITH page_windows AS (

    SELECT
        client_hash_id,
        content_hash_id,

        -- Total feature-window demand: Mar 1-15
        SUM(CASE
            WHEN report_date BETWEEN DATE '2026-03-01'
                                 AND DATE '2026-03-15'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_pre15,

        -- Earlier part of recent-history comparison
        SUM(CASE
            WHEN report_date BETWEEN DATE '2026-03-02'
                                 AND DATE '2026-03-08'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_prev7,

        -- Most recent information available before decision
        SUM(CASE
            WHEN report_date BETWEEN DATE '2026-03-09'
                                 AND DATE '2026-03-15'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_recent7,

        -- FUTURE OUTCOME: never used by the rule
        SUM(CASE
            WHEN report_date BETWEEN DATE '2026-03-17'
                                 AND DATE '2026-03-31'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_next15

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM page_windows
WHERE impressions_pre15 >= 100
""").df()

print("Eligible pages:", len(page_frame))

page_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible pages: 77540


,client_hash_id,content_hash_id,impressions_pre15,impressions_prev7,impressions_recent7,impressions_next15
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,111.0,43.0,68.0,66.0
1,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,219.0,140.0,75.0,673.0
2,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,1494.0,521.0,973.0,1523.0
3,client_62f4a7e64f5e0096,content_d49a012dcb924e31,246.0,143.0,98.0,73.0
4,client_62f4a7e64f5e0096,content_614baf2af4330bd7,413.0,211.0,181.0,329.0


In [7]:
import numpy as np
import pandas as pd

audit_df = page_frame.copy()

# Past-only signal
audit_df["recent_trend_pct"] = (
    100 *
    (
        audit_df["impressions_recent7"]
        - audit_df["impressions_prev7"]
    )
    / audit_df["impressions_prev7"].replace(0, np.nan)
)

# Future outcome used ONLY for evaluation
audit_df["future_ratio"] = (
    audit_df["impressions_next15"]
    / audit_df["impressions_pre15"]
)

audit_df["is_declining_next15d"] = (
    audit_df["future_ratio"] < 0.80
).astype(int)

print(
    "Future decline proxy base rate:",
    f"{audit_df['is_declining_next15d'].mean():.1%}"
)

Future decline proxy base rate: 33.4%


### Signal 1 — Recent impression trend

**Expectation:** pages showing a stronger impression decline immediately
before the decision point may be more likely to decline during the later
outcome window.

This signal is calculated entirely from observations before March 16.

In [8]:
trend_bins = [
    -np.inf,
    -40,
    -20,
    0,
    20,
    np.inf
]

trend_labels = [
    "< -40%",
    "-40% to -20%",
    "-20% to 0%",
    "0% to +20%",
    "> +20%"
]

audit_df["trend_bucket"] = pd.cut(
    audit_df["recent_trend_pct"],
    bins=trend_bins,
    labels=trend_labels
)

trend_audit = (
    audit_df
    .groupby("trend_bucket", observed=True)
    .agg(
        n=("is_declining_next15d", "size"),
        future_decline_rate=("is_declining_next15d", "mean")
    )
    .reset_index()
)

trend_audit["future_decline_rate"] *= 100

trend_audit.round(2)

,trend_bucket,n,future_decline_rate
0,< -40%,11002,59.97
1,-40% to -20%,13041,43.98
2,-20% to 0%,16371,33.59
3,0% to +20%,11988,28.95
4,> +20%,23073,19.71


Verdict: CONFIRMED.

Recent impression trend shows a clear directional relationship with the later decline proxy. Pages with more than a 40% pre-decision decline had the highest observed future-decline rate, while pages with more than 20% growth had the lowest. I will therefore use recent decline severity as the main risk signal in my baseline rule.

### Signal 2 — Search volume / existing exposure

**Expectation:** impression volume may help distinguish higher-value review
opportunities because a decline on a highly visible page places more existing
search exposure at stake.

Volume is also a signal used in FlyRank-style opportunity / quick-win logic.

In [9]:
volume_bins = [
    100,
    300,
    1000,
    3000,
    10000,
    np.inf
]

volume_labels = [
    "100-299",
    "300-999",
    "1k-3k",
    "3k-10k",
    "10k+"
]

audit_df["volume_bucket"] = pd.cut(
    audit_df["impressions_pre15"],
    bins=volume_bins,
    labels=volume_labels,
    right=False
)

volume_audit = (
    audit_df
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("is_declining_next15d", "size"),
        future_decline_rate=("is_declining_next15d", "mean")
    )
    .reset_index()
)

volume_audit["future_decline_rate"] *= 100

volume_audit.round(2)

,volume_bucket,n,future_decline_rate
0,100-299,24582,33.58
1,300-999,25970,33.02
2,1k-3k,16902,32.84
3,3k-10k,8210,34.25
4,10k+,1876,38.91


Verdict: MIXED.

Impression volume does not show a strong monotonic relationship with the future-decline proxy: the first three buckets have very similar decline rates, although the 10k+ group shows a higher rate. I therefore will not treat volume as a direct risk predictor. Instead, I will use it as an opportunity-weighting signal so that comparable declines on pages with greater existing search exposure receive higher review priority.

## 1. My rule and its reason codes

Signal audit

Recent impression trend — CONFIRMED.

The observed future-decline rate increased consistently as the pre-decision trend became worse. Pages with a recent decline greater than 40% had a 59.97% future-decline rate, compared with 19.71% among pages with recent growth above 20%. I therefore use recent decline severity as the main risk signal.


Impression volume — MIXED.

Future-decline rates were similar across the 100–3,000 impression buckets, although the highest-volume group had a somewhat higher observed decline rate. I therefore do not treat volume as a direct predictor of decline. Instead, I use it as an opportunity weight because a comparable decline on a high-exposure page places more existing search visibility at stake.

Baseline rule

I rank pages that have experienced more than a 20% decline in recent impressions before the decision point. The score increases with both the severity of the decline and the page's existing impression volume. Impression volume is log-transformed so that extremely large pages do not dominate the queue solely because of scale.

Reason code: `RECENT_DECLINE_WITH_DEMAND`

Action: `review_for_refresh`

The action means that the page should receive human review; it does not imply that refreshing the page is proven to improve performance.

In [10]:
baseline_df = audit_df.copy()

# Main risk signal:
# positive values mean stronger recent decline
baseline_df["decline_severity"] = (
    -baseline_df["recent_trend_pct"]
).clip(lower=0)

# Opportunity weighting:
# log keeps very large pages from dominating completely
baseline_df["volume_weight"] = np.log1p(
    baseline_df["impressions_pre15"]
)

# One transparent baseline score
baseline_df["baseline_score"] = (
    baseline_df["decline_severity"]
    * baseline_df["volume_weight"]
)

# Only pages with >20% recent decline enter the action queue
baseline_df["eligible_for_review"] = (
    baseline_df["recent_trend_pct"] < -20
)

baseline_df["reason_code"] = np.where(
    baseline_df["eligible_for_review"],
    "RECENT_DECLINE_WITH_DEMAND",
    "NONE"
)

baseline_df["action"] = np.where(
    baseline_df["eligible_for_review"],
    "review_for_refresh",
    "no_action"
)

print(
    "Pages eligible for review:",
    baseline_df["eligible_for_review"].sum()
)

Pages eligible for review: 23956


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
baseline_queue = (
    baseline_df[
        baseline_df["eligible_for_review"]
    ]
    .copy()
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .reset_index(drop=True)
)

baseline_queue["rank"] = (
    np.arange(len(baseline_queue)) + 1
)

queue_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "impressions_pre15",
    "recent_trend_pct",
    "reason_code",
    "action",
]

baseline_queue[queue_cols].head(20)


,rank,client_hash_id,content_hash_id,baseline_score,impressions_pre15,recent_trend_pct,reason_code,action
0,1,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1133.048761,83772.0,-99.952554,RECENT_DECLINE_WITH_DEMAND,review_for_refresh
1,2,client_62f4a7e64f5e0096,content_34a70fea29d15f24,956.783885,73639.0,-85.374203,RECENT_DECLINE_WITH_DEMAND,review_for_refresh
2,3,client_62f4a7e64f5e0096,content_9e0a8a913953b8d3,913.157910,12020.0,-97.202259,RECENT_DECLINE_WITH_DEMAND,review_for_refresh
3,4,client_62f4a7e64f5e0096,content_945d6ff91386c817,907.851901,49314.0,-84.013815,RECENT_DECLINE_WITH_DEMAND,review_for_refresh
4,5,client_62f4a7e64f5e0096,content_0bca6d9a85a9b408,896.015977,15965.0,-92.580689,RECENT_DECLINE_WITH_DEMAND,review_for_refresh
5,6,client_e5c2aa26a8598242,content_8abf2671c081e29e,894.637706,14244.0,-93.540634,RECENT_DECLINE_WITH_DEMAND,review_for_refresh
6,7,client_62f4a7e64f5e0096,content_0c5606abaaab3178,881.689312,27715.0,-86.188617,RECENT_DECLINE_WITH_DEMAND,review_for_refresh
7,8,client_62f4a7e64f5e0096,content_1ff6231687184dec,880.976940,12634.0,-93.282069,RECENT_DECLINE_WITH_DEMAND,review_for_refresh
8,9,client_62f4a7e64f5e0096,content_2d3ea336a4467aa5,860.062956,9307.0,-94.112903,RECENT_DECLINE_WITH_DEMAND,review_for_refresh
9,10,client_0fa64a184f18a4a0,content_fe8328ab251759cc,856.080985,5248.0,-99.941827,RECENT_DECLINE_WITH_DEMAND,review_for_refresh


In [12]:
K = 20

top20 = baseline_queue.head(K)

baseline_precision_at_20 = (
    top20["is_declining_next15d"].mean()
)

print(
    f"Baseline Precision@20: "
    f"{baseline_precision_at_20:.3f}"
)

print(
    f"Future-decline positives in top 20: "
    f"{top20['is_declining_next15d'].sum()} / {K}"
)

Baseline Precision@20: 1.000
Future-decline positives in top 20: 20 / 20


In [13]:
for k in [20, 50, 100, 200, 500]:
    if len(baseline_queue) >= k:
        p_at_k = (
            baseline_queue
            .head(k)["is_declining_next15d"]
            .mean()
        )
        print(f"Precision@{k}: {p_at_k:.3f}")

Precision@20: 1.000
Precision@50: 0.920
Precision@100: 0.900
Precision@200: 0.840
Precision@500: 0.788


The baseline achieved Precision@20 = 1.000 on this March development slice, meaning all 20 highest-ranked pages met the future-decline proxy. This is a strong observed result, but it should not be interpreted as perfect generalization because the rule and thresholds were developed and inspected on the same mid-panel slice.

In [14]:
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/flyrank-intern/work/outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    OUTPUT_DIR / "baseline_action_score.csv"
)

baseline_queue[queue_cols].to_csv(
    output_path,
    index=False
)

print("Queue written to:", output_path)

Queue written to: /content/flyrank-intern/work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
baseline_queue[[
    "rank",
    "content_hash_id",
    "baseline_score",
    "impressions_pre15",
    "recent_trend_pct",
    "is_declining_next15d",
    "action",
]].head(20)

,rank,content_hash_id,baseline_score,impressions_pre15,recent_trend_pct,is_declining_next15d,action
0,1,content_9c057b66c30a3abb,1133.048761,83772.0,-99.952554,1,review_for_refresh
1,2,content_34a70fea29d15f24,956.783885,73639.0,-85.374203,1,review_for_refresh
2,3,content_9e0a8a913953b8d3,913.157910,12020.0,-97.202259,1,review_for_refresh
3,4,content_945d6ff91386c817,907.851901,49314.0,-84.013815,1,review_for_refresh
4,5,content_0bca6d9a85a9b408,896.015977,15965.0,-92.580689,1,review_for_refresh
5,6,content_8abf2671c081e29e,894.637706,14244.0,-93.540634,1,review_for_refresh
6,7,content_0c5606abaaab3178,881.689312,27715.0,-86.188617,1,review_for_refresh
7,8,content_1ff6231687184dec,880.976940,12634.0,-93.282069,1,review_for_refresh
8,9,content_2d3ea336a4467aa5,860.062956,9307.0,-94.112903,1,review_for_refresh
9,10,content_fe8328ab251759cc,856.080985,5248.0,-99.941827,1,review_for_refresh


| Rank | Action             | Reason                                         | Confidence note                                                                     | What could make it wrong                                                             |
| ---: | ------------------ | ---------------------------------------------- | ----------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------ |
|    1 | Review for refresh | 83,772 impressions and a 99.95% recent decline | Very high — extreme deterioration plus very high exposure                           | Temporary tracking/indexing issue or short-lived demand shock                        |
|    2 | Review for refresh | 73,639 impressions and an 85.37% decline       | High — very large exposure, though decline is less severe than many other top picks | Volume weighting may be lifting it above pages with stronger deterioration           |
|    3 | Review for refresh | 12,020 impressions and a 97.20% decline        | Very high — near-total recent collapse                                              | Seasonal demand or temporary SERP/indexing change                                    |
|    4 | Review for refresh | 49,314 impressions and an 84.01% decline       | High — major exposure at risk                                                       | High volume may be contributing more to rank than decline severity                   |
|    5 | Review for refresh | 15,965 impressions and a 92.58% decline        | Very high — severe decline with meaningful demand                                   | Search demand for the topic may have fallen independently of content quality         |
|    6 | Review for refresh | 14,244 impressions and a 93.54% decline        | Very high                                                                           | Temporary technical/search visibility issue rather than refresh need                 |
|    7 | Review for refresh | 27,715 impressions and an 86.19% decline       | High                                                                                | External demand or SERP changes may explain the loss                                 |
|    8 | Review for refresh | 12,634 impressions and a 93.28% decline        | Very high                                                                           | Recent decline could be temporary rather than persistent                             |
|    9 | Review for refresh | 9,307 impressions and a 94.11% decline         | Very high                                                                           | Topic seasonality or indexing instability                                            |
|   10 | Review for refresh | 5,248 impressions and a 99.94% decline         | Very high on decline, moderate on opportunity                                       | Near-total fall could reflect tracking/indexing failure rather than content weakness |
|   11 | Review for refresh | 4,962 impressions and a 99.90% decline         | Very high on decline                                                                | Same: extreme collapse may be technical rather than editorial                        |
|   12 | Review for refresh | 9,103 impressions and a 92.81% decline         | Very high                                                                           | Search-demand shift could make refresh ineffective                                   |
|   13 | Review for refresh | 19,811 impressions and an 85.49% decline       | High                                                                                | Volume weighting may be responsible for its high rank                                |
|   14 | Review for refresh | 18,917 impressions and an 85.80% decline       | High                                                                                | Similar risk: strong volume but less severe decline than many lower-volume pages     |
|   15 | Review for refresh | 36,945 impressions and an 80.28% decline       | High opportunity, slightly lower risk confidence                                    | High exposure strongly boosts score despite the mildest decline in the top 20        |
|   16 | Review for refresh | 7,568 impressions and a 93.13% decline         | Very high                                                                           | Could reflect temporary demand or SERP volatility                                    |
|   17 | Review for refresh | 8,975 impressions and a 90.96% decline         | Very high                                                                           | Decline may not be caused by content freshness                                       |
|   18 | Review for refresh | 6,600 impressions and a 94.13% decline         | Very high                                                                           | Short-term disruption may reverse without intervention                               |
|   19 | Review for refresh | 5,940 impressions and a 94.79% decline         | Very high                                                                           | Could be seasonal or technical rather than editorial                                 |
|   20 | Review for refresh | 4,266 impressions and a 97.43% decline         | Very high on decline, lower opportunity                                             | Severe decline but smaller exposure than most top-ranked pages                       |


## 4. Weak picks + leakage check

Rank 2, 4, 15 are weak picks. Because volume was only MIXED as a future-decline signal, and these rows have somewhat milder decline than many others but get pushed upward by huge impression volume.

In [16]:
BASELINE_INPUTS = [
    "recent_trend_pct",
    "impressions_pre15"
]

FORBIDDEN = {
    "impressions_next15",
    "future_ratio",
    "is_declining_next15d",
    "decline_ratio"
}

leaks = FORBIDDEN.intersection(BASELINE_INPUTS)

print("Forbidden inputs found:", leaks)

assert len(leaks) == 0

print("✓ Baseline uses pre-decision inputs only.")


Forbidden inputs found: set()
✓ Baseline uses pre-decision inputs only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.